# v29 — power и flip-TTA против v25

**Restart Kernel → Run All** в прежнем research `.venv`. Сравниваются ровно три фиксированные системы: `V25_control`, `power_50_l50`, `flip_mvp_w75_l50`.

Это явно разрешённое продолжение исследования после просмотра v28 на уже наблюдавшихся development-данных, не новый независимый тест. Параметры не подбираются, модель не обучается, порог и кандидаты прежние. MVP автоматически не переключается.

In [1]:
from pathlib import Path
import os, sys, subprocess
# Отключаем нативную телеметрию до первого импорта ONNX Runtime.
if 'onnxruntime' in sys.modules and os.environ.get('ORT_DISABLE_TELEMETRY') != '1':
    raise RuntimeError('ONNX Runtime уже загружен. Выполни Restart Kernel → Run All')
os.environ['ORT_DISABLE_TELEMETRY'] = '1'
candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/family_finalists.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из исследовательского Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import Markdown, display
from training import family_finalists as experiment
RUN_NAME = 'family_finalists_v1'
SOURCE_RUN = 'multi_hypothesis_v1'
ALLOW_OUTER_EVALUATION = True
print('Kernel:', sys.executable, '\nRun:', RUN_NAME)
display(experiment.systems())
print('CPU; без обучения и лимита времени; calibration-кэш уже готов; validation flip сохраняется блоками')

Kernel: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python 
Run: family_finalists_v1


[{'name': 'V25_control', 'family': 'control', 'lambda': 0.5},
 {'name': 'power_50_l50', 'family': 'power', 'gamma': 0.5, 'lambda': 0.5},
 {'name': 'flip_mvp_w75_l50',
  'family': 'flip',
  'scope': 'mvp',
  'flip_weight': 0.75,
  'lambda': 0.5}]

CPU; без обучения и лимита времени; calibration-кэш уже готов; validation flip сохраняется блоками


## 1. Проверки компонентов

Проверяются фиксированный список, запрет дополнительного подбора, независимость query, сохранность кандидатов, экспорт и продолжение после ошибки.

In [2]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_family_finalists.py', 'tests/test_multi_hypothesis.py', 'tests/test_dual_role_inference.py'], check=True, env={**os.environ, 'ORT_DISABLE_TELEMETRY': '1'})

......................................................                   [100%]
54 passed in 10.60s


CompletedProcess(args=['/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python', '-m', 'pytest', '-q', 'tests/test_family_finalists.py', 'tests/test_multi_hypothesis.py', 'tests/test_dual_role_inference.py'], returncode=0)

## 2. Проверка завершённого v28 и фиксация сравнения

Проверяем, что power и flip — именно победители соответствующих семейств по calibration. Другой список не принимается. Старые runs, веса, исходная validation и bbox не изменяются.

In [3]:
context = experiment.prepare(RUN_NAME, source_run=SOURCE_RUN)
print('Results:', context['output'])
display(context['manifest']['finalists_plan'])

Preflight OK: three fixed finalists, unchanged candidates, no training or automatic promotion
Results: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_29_family_finalists/runs/family_finalists_v1


{'source_directory': '/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_28_multi_hypothesis/runs/multi_hypothesis_v1',
 'source_signature': 'e96bc83f77ea2ba554867164f83335eb647eac4dcefe196ec09424ea5d183495',
 'source_result_sha256': '2accf0999c1ef4f21043032dad78e579645f7f5006df2237e22cac70f49ec68f',
 'systems': [{'name': 'V25_control', 'family': 'control', 'lambda': 0.5},
  {'name': 'power_50_l50', 'family': 'power', 'gamma': 0.5, 'lambda': 0.5},
  {'name': 'flip_mvp_w75_l50',
   'family': 'flip',
   'scope': 'mvp',
   'flip_weight': 0.75,
   'lambda': 0.5}],
 'calibration_reports': {'V25_control': {'system': {'name': 'V25_control',
    'family': 'control',
    'lambda': 0.5},
   'split': 'calibration',
   'threshold': 0.534365177154541,
   'ranking': {'n_scored': 246,
    'n_openset_excluded': 61,
    'mAP@10': 0.826409928470033,
    'Rank-1': 0.8414634146341463,
    'Rank-5': 0.9024390243902439},
   'candidates': {'TP': 1

## 3. Calibration replay → три validation-экспорта

Повторный расчёт трёх прежних calibration-результатов должен совпасть точно, без нового извлечения признаков. Затем v25, power и flip сравниваются на validation.

Видны этап, система, время и прогресс TTA. После прерывания **Restart Kernel → Run All с тем же RUN_NAME**. Не меняй код между частями запуска.

In [4]:
results = experiment.run(context, allow_outer=ALLOW_OUTER_EVALUATION)
print('Status:', results['status'], '| promoted:', results['promoted'])

[START] verify_calibration | elapsed 0.00 h
CALIBRATION REPLAY V25_control: 0.826410 exact
CALIBRATION REPLAY power_50_l50: 0.833388 exact
CALIBRATION REPLAY flip_mvp_w75_l50: 0.828729 exact
[DONE] verify_calibration | 0.0 min

FINALIST 1/3: V25_control | elapsed 11.7s
[START] validation_V25_control | elapsed 0.00 h
VALIDATION V25_control: mAP=0.828957; candidates unchanged
[DONE] validation_V25_control | 0.0 min

FINALIST 2/3: power_50_l50 | elapsed 12.7s
[START] validation_power_50_l50 | elapsed 0.00 h
VALIDATION power_50_l50: mAP=0.814838; candidates unchanged
[DONE] validation_power_50_l50 | 0.0 min

FINALIST 3/3: flip_mvp_w75_l50 | elapsed 13.7s
[START] flip_validation | elapsed 0.00 h
FLIP validation: 128/1205 images
FLIP validation: 256/1205 images
FLIP validation: 384/1205 images
FLIP validation: 512/1205 images
FLIP validation: 640/1205 images
FLIP validation: 768/1205 images
FLIP validation: 896/1205 images
FLIP validation: 1024/1205 images
FLIP validation: 1152/1205 images
F

## 4. Отчёт

При ошибке отчёт сохраняется частично; эту ячейку можно выполнить отдельно. Самый высокий наблюдавшийся validation-результат не объявляется независимой оценкой скрытого теста.

In [5]:
display(Markdown((context['output'] / 'REPORT.md').read_text()))
print('Reports, exports and errors:', context['output'])

# v29 — два фиксированных финалиста против v25

Статус: complete
Это post-hoc сравнение на уже наблюдавшейся development validation, не независимый тест.
Нового перебора параметров, обучения, изменения порога и автоматического продвижения нет.

| Вариант | Calibration v28 | Validation mAP@10 | Rank-1 | F1 | TNR |
|---|---:|---:|---:|---:|---:|
| V25_control | 0.826410 | 0.828957 | 0.829960 | 0.795455 | 0.709677 |
| power_50_l50 | 0.833388 | 0.814838 | 0.805668 | 0.795455 | 0.709677 |
| flip_mvp_w75_l50 | 0.828729 | 0.819686 | 0.817814 | 0.795455 | 0.709677 |

power_50_l50: ΔmAP -0.014119; AP лучше/хуже/равно 14/21/212.

flip_mvp_w75_l50: ΔmAP -0.009272; AP лучше/хуже/равно 16/23/208.

Лучший наблюдавшийся результат из этих трёх: V25_control.
Это описание development-результата, не подтверждение превосходства на скрытом тесте и не смена MVP.

Порог, raw R1 candidate/confidence, bbox, evaluator и исходная validation неизменны.
Контроль и power: реальные 2048 признаков; flip: original2048 + flipped2048. NPY replay проверяется.
Кандидаты всех систем обязаны побайтно совпасть с v25. Каждый query независим от остальных.
v28 calibration flip-кэш переиспользуется; заново извлекаются только validation flip-признаки.
Готовые задачи/блоки возобновляются. Неполное сравнение не объявляется завершённым.
Защищённые источники неизменны: True.
Время текущего вызова: 2.3 мин; не замер конкурсной скорости.


Reports, exports and errors: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_29_family_finalists/runs/family_finalists_v1
